# Dopamine PINN: extension experiments

Companion to `dopamine_PINN.ipynb` (main pipeline) and `dopamine_PINN_tuning.ipynb`.
Five self-contained experiments that test the paper's claims beyond the single
linear, homogeneous, one-draw setting. Every experiment uses the final inverse
setting of the paper: MLP [3, 64, 64, 64, 64, 1] (tanh, raw inputs), Adam 20k
iterations at lr 1e-3 then L-BFGS (max 2000, tol 1e-9), N_r = 30k (Latin
Hypercube), N_i = 400, N_b = 2000, weights W_RES = 3000, W_IC = 1, W_BC = 100,
W_D = 5, 2% noise, observations in t in [0.5, 45] ms.

| Toggle | Experiment | PINN runs | A100 time (est.) |
|---|---|---|---|
| `RUN_NOISE_DRAWS` | **E1** 20 noise realizations: PINN vs reference-estimator spread, 95% interval coverage | 20 | ~1.5 h |
| `RUN_TRUTHFREE` | **E2** truth-free data-weight selection: discrepancy principle and K-fold cross-validation over W_D | 18 + 24 | ~3 h |
| `RUN_MM` | **E3** Michaelis-Menten uptake in a 50 ms and a 500 ms record: when are Vmax and Km identifiable; linear model fitted to MM data | 6 | ~40 min |
| `RUN_DFIELD` | **E4** spatially varying D(x, y) recovered by a second network | 2 | ~20 min |
| `RUN_OBSTACLES` | **E5** extracellular space with cellular obstacles; homogeneous fit gives the apparent tortuosity | 3 | ~25 min |

The FD reference solver here is written in JAX (flux form, optional obstacle mask,
pluggable uptake term), so the reference estimator gets exact Jacobians by forward-mode
autodiff and runs on the GPU. For constant D it is the same scheme as the main
notebook (mirror Neumann walls); cell 3 checks the two agree.

**Results are saved after every run** (to Google Drive if `USE_DRIVE = True`). A rerun
skips runs already in the JSON, so a Colab disconnect costs at most one run. Each
experiment writes `E*_*.json` and `E*_*.png`; the last cell zips and downloads them.

## 0. Install dependencies (same pins as the other notebooks)

In [ ]:
import os
from pathlib import Path

SENTINEL = Path('/tmp/jax_pinn_deps_ready_v2')
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB and not SENTINEL.exists():
    # Pinned, mutually compatible versions (an unpinned flax/jax upgrade breaks
    # `from flax import nnx`).
    rc = os.system('pip -q install '
                   '"jax[cuda12]==0.10.2" '
                   '"flax==0.12.5" '
                   '"optax>=0.2.3" '
                   '"jaxopt>=0.8" '
                   'scipy matplotlib')
    if rc != 0:
        raise RuntimeError('pip install failed - see the output above.')
    SENTINEL.touch()
    print('Install complete. Restarting kernel; then Runtime -> Run all again.')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies ready.')

In [ ]:
import os, time, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc, chi2
from scipy.optimize import least_squares

import jax
import jax.numpy as jnp
from jax import lax
from flax import nnx
import optax
from jaxopt import LBFGS

jax.config.update('jax_enable_x64', True)
print(f'JAX {jax.__version__}, devices: {jax.devices()}')

## 1. Toggles and parameters

In [ ]:
# ---- Toggles -----------------------------------------------------------------
QUICK           = False   # tiny smoke test (minutes, CPU is fine); not paper-quality
USE_DRIVE       = True    # save to Google Drive so a disconnect does not lose runs
RUN_NOISE_DRAWS = False   # E1  20 noise realizations + coverage          (~1.5 h A100)
RUN_TRUTHFREE   = False   # E2  truth-free W_D selection                  (~3 h A100)
RUN_MM          = True    # E3  Michaelis-Menten uptake, 50 and 500 ms    (~40 min)
RUN_DFIELD      = True    # E4  spatially varying D(x, y)                 (~20 min)
RUN_OBSTACLES   = True    # E5  obstacles / tortuosity                    (~25 min)

# ---- Physics (as in the paper) ---------------------------------------------------
D_TRUE, K_TRUE = 0.32, 0.020       # um^2/ms, 1/ms
L, T, SIGMA, C0 = 5.0, 50.0, 0.5, 1.0
D_INIT, K_INIT = 0.30, 0.016       # offset initial guesses (-6.25%, -20%)
NOISE_PCT = 2.0
T_OBS = (0.5, 0.9 * T)
N_OBS = 400

# ---- Network and training (final inverse setting of the paper) -------------------
LAYERS      = [3] + [64] * 4 + [1]
ADAM_ITERS  = 20_000
LBFGS_ITERS = 2_000
LR          = 1e-3
N_CHUNK     = 50
N_DOMAIN    = 30_000
N_INITIAL   = 400
N_BOUNDARY  = 2_000
W_RES, W_IC, W_BC, W_D = 3000.0, 1.0, 100.0, 5.0
SEED = 1234

# ---- FD reference solver ---------------------------------------------------------
NX_FD    = 81        # nodes per side (dx = 0.0625 um), as in the main notebook
DT_STORE = 0.1       # ms between stored frames (observations interpolate linearly)

if QUICK:
    ADAM_ITERS, LBFGS_ITERS, N_CHUNK = 600, 100, 50
    N_DOMAIN, N_INITIAL, N_BOUNDARY = 1_500, 200, 400
    N_OBS = 200
    NX_FD, DT_STORE = 41, 0.25
    print('[QUICK MODE] reduced settings - not paper-quality.')
assert ADAM_ITERS % N_CHUNK == 0

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/dopamine_PINN_extensions')
else:
    OUT_DIR = Path('extensions_out')
if QUICK:
    OUT_DIR = OUT_DIR / 'quick'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Results -> {OUT_DIR.resolve()}')

## 2. FD reference solver (JAX) and reference estimator

Node-centred flux form on an `nx x nx` grid over $[-L/2, L/2]^2$:
$\partial_t C_{ij} = [F^x_{i+1/2,j} - F^x_{i-1/2,j}]/(w_i\,\Delta x^2) + [\ldots]^y - s(C_{ij})$,
with face flux $F^x_{i+1/2,j} = \bar D\,(C_{i+1,j} - C_{ij})$, $\bar D$ the mean of the two
nodal values, and $w_i = 1/2$ on the walls (half control volume, zero wall flux). For
constant $D$ this is the mirror-Neumann scheme of the main notebook. An obstacle mask
sets the flux through every face that touches a solid node to zero. The time step is
fixed by a static cap `D_cap` on $D$ so the solver can be differentiated with respect to
$D$ (forward mode), and the reference estimator gets exact Jacobians.

In [ ]:
def gaussian_ic(nx, amp=C0):
    x = np.linspace(-L / 2, L / 2, nx)
    X, Y = np.meshgrid(x, x, indexing='ij')
    return amp * np.exp(-(X**2 + Y**2) / (2 * SIGMA**2))


def make_fd_solver(nx=None, D_cap=0.64, mask=None, T_end=None, dt_store=None):
    """Returns solve(D_nodes, sink, C_init) -> frames (n_frames + 1, nx, nx).

    Frames are stored at t = j * dt_store up to T_end (defaults DT_STORE, T).
    D_nodes is a scalar or (nx, nx) array with D <= D_cap; sink(C) is the uptake
    rate (e.g. k * C).
    """
    nx = NX_FD if nx is None else nx
    T_end = T if T_end is None else T_end
    dt_store = DT_STORE if dt_store is None else dt_store
    dx = L / (nx - 1)
    dt_max = 0.95 * dx**2 / (4.0 * D_cap)
    stride = int(np.ceil(dt_store / dt_max))
    dt = dt_store / stride
    n_frames = int(round(T_end / dt_store))
    m = np.ones((nx, nx)) if mask is None else mask.astype(float)
    mfx = jnp.asarray(m[1:, :] * m[:-1, :])
    mfy = jnp.asarray(m[:, 1:] * m[:, :-1])
    w = np.ones(nx); w[0] = w[-1] = 0.5
    ivx = jnp.asarray(1.0 / (dx**2 * w))[:, None]
    ivy = jnp.asarray(1.0 / (dx**2 * w))[None, :]
    mj = jnp.asarray(m)

    def solve(D_nodes, sink, C_init):
        Dn = jnp.broadcast_to(jnp.asarray(D_nodes, jnp.float64), (nx, nx))
        Dfx = 0.5 * (Dn[1:, :] + Dn[:-1, :]) * mfx
        Dfy = 0.5 * (Dn[:, 1:] + Dn[:, :-1]) * mfy

        def step(_, C):
            Fx = Dfx * (C[1:, :] - C[:-1, :])
            Fy = Dfy * (C[:, 1:] - C[:, :-1])
            div = ((jnp.pad(Fx, ((0, 1), (0, 0))) - jnp.pad(Fx, ((1, 0), (0, 0)))) * ivx
                   + (jnp.pad(Fy, ((0, 0), (0, 1))) - jnp.pad(Fy, ((0, 0), (1, 0)))) * ivy)
            return C + dt * (div - sink(C)) * mj

        def frame(C, _):
            C = lax.fori_loop(0, stride, step, C)
            return C, C

        C_init = jnp.asarray(C_init) * mj
        _, F = lax.scan(frame, C_init, None, length=n_frames)
        return jnp.concatenate([C_init[None], F])

    solve.nx, solve.dx, solve.dt, solve.mask = nx, dx, dt, m
    solve.dt_store, solve.T_end = dt_store, T_end
    return solve


def interp_frames(frames, dx, xq, yq, tq, dt_store=None):
    """Trilinear interpolation of stored frames at (x, y, t) points (JAX)."""
    nf, nx, _ = frames.shape
    ft = tq / (DT_STORE if dt_store is None else dt_store)
    it = jnp.clip(jnp.floor(ft).astype(int), 0, nf - 2); a = ft - it
    fx = (xq + L / 2) / dx
    ix = jnp.clip(jnp.floor(fx).astype(int), 0, nx - 2); b = fx - ix
    fy = (yq + L / 2) / dx
    iy = jnp.clip(jnp.floor(fy).astype(int), 0, nx - 2); c = fy - iy

    def bil(ti):
        return ((1 - b) * (1 - c) * frames[ti, ix, iy] + b * (1 - c) * frames[ti, ix + 1, iy]
                + (1 - b) * c * frames[ti, ix, iy + 1] + b * c * frames[ti, ix + 1, iy + 1])
    return (1 - a) * bil(it) + a * bil(it + 1)


def ref_fit(predict, theta0, C_obs, sigma, lb, ub):
    """Reference estimator: bounded non-linear least squares through the FD solver.

    predict(theta) -> concentrations at the observation points (JAX, differentiable).
    Returns the estimate, the Gauss-Newton (Laplace) covariance of theta, the residual
    RMS and chi^2 statistics. The Jacobian is exact (forward-mode autodiff).
    """
    f = jax.jit(predict)
    J = jax.jit(jax.jacfwd(predict))
    th0 = np.clip(np.asarray(theta0, float), np.asarray(lb) + 1e-9, np.asarray(ub) - 1e-9)
    r = least_squares(lambda th: np.asarray(f(jnp.asarray(th))) - C_obs, th0,
                      jac=lambda th: np.asarray(J(jnp.asarray(th))),
                      bounds=(lb, ub), x_scale='jac', xtol=1e-10, ftol=1e-12, gtol=1e-10)
    Jm = r.jac
    cov = np.linalg.pinv(Jm.T @ Jm) * sigma**2
    n, p = C_obs.size, len(theta0)
    rss = float(np.sum(r.fun**2))
    return {'theta': r.x, 'cov': cov, 'rms': float(np.sqrt(rss / n)),
            'chi2': rss / sigma**2, 'dof': n - p,
            'p_chi2': float(chi2.sf(rss / sigma**2, n - p)),
            'aic': n * np.log(rss / n) + 2 * p, 'nfev': int(r.nfev),
            'at_bound': bool(np.any(np.isclose(r.x, lb) | np.isclose(r.x, ub)))}


def scattered_obs(frames, solver, seed, n_obs=None, noise_pct=NOISE_PCT, valid=None, t_obs=T_OBS):
    """Uniform (x, y, t) observations with Gaussian noise (draw order as the main notebook).

    valid(x, y) -> bool restricts locations (obstacles); rejected points are redrawn.
    """
    n_obs = N_OBS if n_obs is None else n_obs
    rng = np.random.default_rng(seed)
    if valid is None:
        ox = rng.uniform(-L / 2, L / 2, n_obs); oy = rng.uniform(-L / 2, L / 2, n_obs)
    else:
        ox, oy = np.empty(0), np.empty(0)
        while ox.size < n_obs:
            cx = rng.uniform(-L / 2, L / 2, 4 * n_obs); cy = rng.uniform(-L / 2, L / 2, 4 * n_obs)
            ok = valid(cx, cy)
            ox = np.concatenate([ox, cx[ok]]); oy = np.concatenate([oy, cy[ok]])
        ox, oy = ox[:n_obs], oy[:n_obs]
    ot = rng.uniform(*t_obs, n_obs)
    clean = np.asarray(interp_frames(frames, solver.dx, jnp.asarray(ox), jnp.asarray(oy), jnp.asarray(ot),
                                     solver.dt_store))
    noisy = clean + rng.normal(0.0, noise_pct / 100 * C0, n_obs)
    return {'x_d': ox, 'y_d': oy, 't_d': ot, 'C_d': noisy, 'C_clean': clean}


# ---- check against the numpy solver of the main notebook ------------------------
def fd_reference_np(D=D_TRUE, k=K_TRUE, nx=81, dt_store=0.02):
    dx = L / (nx - 1)
    nt = int(np.ceil(T / (0.95 / (2.0 * D * (2.0 / dx**2))))) + 1
    dt = T / (nt - 1)
    C = gaussian_ic(nx)
    for n in range(1, nt):
        lap = np.zeros_like(C)
        lap[1:-1, :] += (C[2:, :] - 2 * C[1:-1, :] + C[:-2, :]) / dx**2
        lap[0, :] += 2 * (C[1, :] - C[0, :]) / dx**2
        lap[-1, :] += 2 * (C[-2, :] - C[-1, :]) / dx**2
        lap[:, 1:-1] += (C[:, 2:] - 2 * C[:, 1:-1] + C[:, :-2]) / dx**2
        lap[:, 0] += 2 * (C[:, 1] - C[:, 0]) / dx**2
        lap[:, -1] += 2 * (C[:, -2] - C[:, -1]) / dx**2
        C = C + D * dt * lap - k * dt * C
    return C


SOLVE_LIN = make_fd_solver(D_cap=0.64)
t0 = time.time()
FR_LIN = jax.block_until_ready(SOLVE_LIN(D_TRUE, lambda C: K_TRUE * C, gaussian_ic(SOLVE_LIN.nx)))
print(f'JAX FD solve: {time.time() - t0:.1f} s (incl. compile), frames {FR_LIN.shape}')
C_np = fd_reference_np(nx=SOLVE_LIN.nx)
print(f'max |JAX - numpy| at t = T: {np.max(np.abs(np.asarray(FR_LIN[-1]) - C_np)):.2e} '
      f'(peak {C_np.max():.4f}; the two differ only in time step)')

## 3. PINN building blocks

One MLP class for the concentration network (and a smaller one for the $D(x, y)$
network of E4). A loss is assembled from a `physics(p, x, y, t, C, grad, lap)` function
that returns the PDE residual at one point; the trainer is the Adam-then-L-BFGS loop of
the paper. Collocation, initial and boundary points can be restricted to the
extracellular space and include obstacle boundaries (E5).

In [ ]:
class MLP(nnx.Module):
    """tanh MLP, float64 parameters (dtype-homogeneous pytree for jaxopt L-BFGS)."""
    def __init__(self, layers, *, rngs, zero_last=False):
        self.n_layers = len(layers) - 1
        for i in range(self.n_layers):
            last = zero_last and i == self.n_layers - 1
            setattr(self, f'lin_{i}', nnx.Linear(
                layers[i], layers[i + 1],
                kernel_init=(nnx.initializers.zeros_init() if last
                             else nnx.initializers.glorot_normal()),
                bias_init=nnx.initializers.zeros_init(),
                param_dtype=jnp.float64, rngs=rngs))

    def __call__(self, z):
        h = z
        for i in range(self.n_layers - 1):
            h = jnp.tanh(getattr(self, f'lin_{i}')(h))
        return getattr(self, f'lin_{self.n_layers - 1}')(h)


GDEF = nnx.split(MLP(LAYERS, rngs=nnx.Rngs(0)))[0]
D_LAYERS = [2, 32, 32, 1]
GDEF_D = nnx.split(MLP(D_LAYERS, rngs=nnx.Rngs(0), zero_last=True))[0]


def init_net(seed):
    return nnx.split(MLP(LAYERS, rngs=nnx.Rngs(seed)))[1]


def net_fn(p, t_scale=1.0):
    """Scalar C(x, y, t). t_scale < 1 maps a long record onto the paper's t range [0, 50]."""
    net = nnx.merge(GDEF, p['net'])
    sc = jnp.array([1.0, 1.0, 1.0]).at[2].set(t_scale)
    return lambda z: net(z * sc)[0]


EX, EY = jnp.array([1.0, 0.0, 0.0]), jnp.array([0.0, 1.0, 0.0])


def point_derivs(f, x, y, t):
    """C, grad (C_x, C_y, C_t) and the Laplacian C_xx + C_yy at one point."""
    z = jnp.stack([x, y, t])
    g = jax.grad(f)
    gz, hx = jax.jvp(g, (z,), (EX,))
    _, hy = jax.jvp(g, (z,), (EY,))
    return f(z), gz, hx[0] + hy[1]


# ---- physics (PDE residuals) ------------------------------------------------------
def phys_linear(p, x, y, t, C, g, lap):
    return g[2] - jnp.exp(p['logD']) * lap + jnp.exp(p['logk']) * C


def phys_mm(p, x, y, t, C, g, lap):
    # |C| keeps the denominator positive if the network dips below zero early on
    Vmax, Km = jnp.exp(p['logVmax']), jnp.exp(p['logKm'])
    return g[2] - jnp.exp(p['logD']) * lap + Vmax * C / (Km + jnp.abs(C))


D_REF_FIELD = D_INIT    # D(x, y) = D_REF_FIELD * exp(dnet(x, y)); dnet starts at 0


def d_field(p):
    dn = nnx.merge(GDEF_D, p['dnet'])
    return lambda xy: D_REF_FIELD * jnp.exp(dn(xy)[0])


def phys_dfield(p, x, y, t, C, g, lap):
    Df = d_field(p)
    xy = jnp.stack([x, y])
    D, gD = Df(xy), jax.grad(Df)(xy)
    return g[2] - (D * lap + gD[0] * g[0] + gD[1] * g[1]) + jnp.exp(p['logk']) * C


def build_loss(physics, w_d=W_D, extra=None):
    """Total loss W_RES L_r + W_IC L_i + W_BC L_b + w_d L_d (+ extra(p, b))."""
    def components(p, b):
        f = net_fn(p, b.get('t_scale', 1.0))
        res = jax.vmap(lambda x, y, t: physics(p, x, y, t, *point_derivs(f, x, y, t)))(
            b['x_r'], b['y_r'], b['t_r'])
        C_ic = jax.vmap(lambda x, y: f(jnp.stack([x, y, 0.0])))(b['x_i'], b['y_i'])
        dn = jax.vmap(lambda x, y, t, nx, ny:
                      jnp.dot(jax.grad(f)(jnp.stack([x, y, t]))[:2], jnp.stack([nx, ny])))(
            b['x_b'], b['y_b'], b['t_b'], b['nx_b'], b['ny_b'])
        C_d = jax.vmap(lambda x, y, t: f(jnp.stack([x, y, t])))(b['x_d'], b['y_d'], b['t_d'])
        return (jnp.mean(res**2), jnp.mean((C_ic - b['C_i'])**2),
                jnp.mean(dn**2), jnp.mean((C_d - b['C_d'])**2))

    def loss(p, b):
        L_r, L_i, L_b, L_d = components(p, b)
        tot = W_RES * L_r + W_IC * L_i + W_BC * L_b + w_d * L_d
        return tot + (extra(p, b) if extra is not None else 0.0)

    loss.components = components
    return loss


def train(params, loss, batch, label='', report=None):
    """Adam (ADAM_ITERS, lr LR) then L-BFGS (LBFGS_ITERS, tol 1e-9), as in the paper."""
    opt = optax.adam(LR)
    state = opt.init(params)

    @jax.jit
    def chunk(p, s, b):
        def body(_, carry):
            p, s, _l = carry
            l, g = jax.value_and_grad(loss)(p, b)
            u, s = opt.update(g, s, p)
            return optax.apply_updates(p, u), s, l
        return lax.fori_loop(0, N_CHUNK, body, (p, s, jnp.zeros((), jnp.float64)))

    t0 = time.time()
    n_chunks = ADAM_ITERS // N_CHUNK
    for c in range(n_chunks):
        params, state, l = chunk(params, state, batch)
        if c == 0 or (c + 1) % 80 == 0 or c == n_chunks - 1:
            extra = report(params) if report else ''
            print(f'    [{label}] step {(c + 1) * N_CHUNK:>5d}  loss {float(l):.4e}  {extra}  '
                  f'{time.time() - t0:.0f}s')
    params = LBFGS(fun=loss, maxiter=LBFGS_ITERS, tol=1e-9).run(params, b=batch).params
    print(f'    [{label}] after L-BFGS: loss {float(loss(params, batch)):.4e}  '
          f'{report(params) if report else ""}  total {time.time() - t0:.0f}s')
    return params


# ---- collocation ------------------------------------------------------------------
def sample_points(n_domain=None, n_initial=None, n_boundary=None, seed=SEED,
                  obstacles=None, n_obstacle_bc=0, T_end=T, amp=C0, t_power=1.0):
    """LHS interior points, uniform IC points, wall points (+ obstacle-boundary points).

    obstacles: list of (xc, yc, R) discs; interior and IC points inside them are
    rejected, and n_obstacle_bc boundary points are spread over the circles in
    proportion to circumference, with the radial unit normal. For a long record
    (T_end > T) t_power > 1 puts more interior points at early times (t = T_end u^p).
    """
    n_domain = N_DOMAIN if n_domain is None else n_domain
    n_initial = N_INITIAL if n_initial is None else n_initial
    n_boundary = N_BOUNDARY if n_boundary is None else n_boundary
    rng = np.random.default_rng(seed)
    outside = (lambda x, y: np.ones_like(x, bool)) if not obstacles else (
        lambda x, y: np.all([(x - a)**2 + (y - b)**2 > R**2 for a, b, R in obstacles], axis=0))

    over = 1.0 if not obstacles else 1.6
    u = qmc.LatinHypercube(d=3, seed=seed).random(int(over * n_domain))
    x_r, y_r, t_r = -L / 2 + L * u[:, 0], -L / 2 + L * u[:, 1], T_end * u[:, 2]**t_power
    keep = outside(x_r, y_r)
    x_r, y_r, t_r = x_r[keep][:n_domain], y_r[keep][:n_domain], t_r[keep][:n_domain]

    x_i, y_i = np.empty(0), np.empty(0)
    while x_i.size < n_initial:
        cx, cy = rng.uniform(-L / 2, L / 2, 2 * n_initial), rng.uniform(-L / 2, L / 2, 2 * n_initial)
        ok = outside(cx, cy)
        x_i, y_i = np.concatenate([x_i, cx[ok]]), np.concatenate([y_i, cy[ok]])
    x_i, y_i = x_i[:n_initial], y_i[:n_initial]
    C_i = amp * np.exp(-(x_i**2 + y_i**2) / (2 * SIGMA**2))

    pe = n_boundary // 4
    s = lambda: rng.uniform(-L / 2, L / 2, pe)
    tt = lambda n: rng.uniform(0, T_end, n)
    edges = [(np.full(pe, -L / 2), s(), tt(pe), -np.ones(pe), np.zeros(pe)),
             (np.full(pe, L / 2), s(), tt(pe), np.ones(pe), np.zeros(pe)),
             (s(), np.full(pe, -L / 2), tt(pe), np.zeros(pe), -np.ones(pe)),
             (s(), np.full(pe, L / 2), tt(pe), np.zeros(pe), np.ones(pe))]
    if obstacles and n_obstacle_bc:
        circ = np.array([R for _, _, R in obstacles])
        counts = np.round(n_obstacle_bc * circ / circ.sum()).astype(int)
        for (a, b_, R), n in zip(obstacles, counts):
            th = rng.uniform(0, 2 * np.pi, n)
            edges.append((a + R * np.cos(th), b_ + R * np.sin(th), tt(n), np.cos(th), np.sin(th)))
    cat = lambda i: np.concatenate([e[i] for e in edges])
    return {'x_r': x_r, 'y_r': y_r, 't_r': t_r, 'x_i': x_i, 'y_i': y_i, 'C_i': C_i,
            'x_b': cat(0), 'y_b': cat(1), 't_b': cat(2), 'nx_b': cat(3), 'ny_b': cat(4)}


def make_batch(pts, obs, t_scale=1.0):
    b = dict(pts)
    b.update({k: obs[k] for k in ('x_d', 'y_d', 't_d', 'C_d')})
    b['t_scale'] = t_scale
    return {k: jnp.asarray(np.asarray(v, np.float64)) for k, v in b.items()}


# ---- bookkeeping ------------------------------------------------------------------
def load_results(name, meta):
    path = OUT_DIR / f'{name}.json'
    if path.exists():
        res = json.loads(path.read_text())
        print(f'Resuming {path.name}: {len(res["runs"])} runs already done.')
        return res
    return {**meta, 'runs': {}}


def save_results(name, res):
    (OUT_DIR / f'{name}.json').write_text(json.dumps(res, indent=2, default=float))


def pct(est, true):
    return 100.0 * (est - true) / true


def lin_predict(obs, solver, amp=C0):
    """theta = (log D, log k) -> FD concentrations at the observation points."""
    ox, oy, ot = (jnp.asarray(obs[k]) for k in ('x_d', 'y_d', 't_d'))
    C_init = gaussian_ic(solver.nx, amp)
    def predict(th):
        fr = solver(jnp.exp(th[0]), lambda C: jnp.exp(th[1]) * C, C_init)
        return interp_frames(fr, solver.dx, ox, oy, ot, solver.dt_store)
    return predict


def lin_ref_fit(obs, solver=None, D_cap=0.64, amp=C0):
    solver = SOLVE_LIN if solver is None else solver
    return ref_fit(lin_predict(obs, solver, amp), np.log([D_INIT, K_INIT]), obs['C_d'],
                   NOISE_PCT / 100 * C0, np.log([0.02, 1e-4]), np.log([0.97 * D_cap, 0.5]))


def finish(name, figs=()):
    print(f'Saved {OUT_DIR / (name + ".json")}' + ''.join(f', {f}' for f in figs))

## E1. Many noise realizations and interval coverage

The paper's headline numbers come from one observation draw (`OBS_SEED = 4001`). Here the
whole experiment is repeated on `N_DRAWS` fresh draws (new locations and noise each, seeds
7001, 7002, ... never used before). For each draw the PINN (network seed 1234) and the
reference estimator are fitted. Outputs: mean error, SD and RMSE of each method, how often
the reference estimator's Laplace 95% interval contains the truth (should be ~95%), and how
often the PINN estimate falls inside that interval. This turns "the PINN error is close to
the data limit" from a single-draw observation into a statistical statement.

In [ ]:
N_DRAWS = 3 if QUICK else 20
DRAW_SEEDS = [7001 + i for i in range(N_DRAWS)]

if not RUN_NOISE_DRAWS:
    print('RUN_NOISE_DRAWS = False, skipping.')
else:
    name = 'E1_noise_draws'
    res = load_results(name, {'draw_seeds': DRAW_SEEDS, 'net_seed': SEED, 'noise_pct': NOISE_PCT,
                              'W': [W_RES, W_IC, W_BC, W_D], 'N_obs': N_OBS})
    pts = sample_points()
    for s in DRAW_SEEDS:
        key = str(s)
        if key in res['runs']:
            continue
        obs = scattered_obs(FR_LIN, SOLVE_LIN, s)
        t0 = time.time()
        rf = lin_ref_fit(obs)
        D_r, k_r = np.exp(rf['theta'])
        sd = np.sqrt(np.diag(rf['cov']))
        p0 = {'net': init_net(SEED), 'logD': jnp.log(D_INIT), 'logk': jnp.log(K_INIT)}
        p = train(p0, build_loss(phys_linear), make_batch(pts, obs), label=f'draw {s}',
                  report=lambda q: f'D {float(jnp.exp(q["logD"])):.4f} k {float(jnp.exp(q["logk"])):.5f}')
        D_p, k_p = float(jnp.exp(p['logD'])), float(jnp.exp(p['logk']))
        z = lambda est, true, s_: abs(np.log(est / true)) / s_      # in Laplace SDs (log space)
        res['runs'][key] = {
            'pinn_D': D_p, 'pinn_k': k_p, 'ref_D': D_r, 'ref_k': k_r,
            'pinn_D_err': pct(D_p, D_TRUE), 'pinn_k_err': pct(k_p, K_TRUE),
            'ref_D_err': pct(D_r, D_TRUE), 'ref_k_err': pct(k_r, K_TRUE),
            'ref_sd_logD': sd[0], 'ref_sd_logk': sd[1],
            'ref_rho': rf['cov'][0, 1] / (sd[0] * sd[1]),
            'ref_covers_D': bool(z(D_r, D_TRUE, sd[0]) <= 1.96),
            'ref_covers_k': bool(z(k_r, K_TRUE, sd[1]) <= 1.96),
            'pinn_in_ref95_D': bool(z(D_p, D_r, sd[0]) <= 1.96),
            'pinn_in_ref95_k': bool(z(k_p, k_r, sd[1]) <= 1.96),
            'truth_in_ellipse_ref': bool(np.log([D_r / D_TRUE, k_r / K_TRUE]) @ np.linalg.solve(
                rf['cov'], np.log([D_r / D_TRUE, k_r / K_TRUE])) <= chi2.ppf(0.95, 2)),
            'pinn_in_ellipse_ref': bool(np.log([D_p / D_r, k_p / k_r]) @ np.linalg.solve(
                rf['cov'], np.log([D_p / D_r, k_p / k_r])) <= chi2.ppf(0.95, 2)),
            'minutes': (time.time() - t0) / 60}
        save_results(name, res)
        r = res['runs'][key]
        print(f'  draw {s}: PINN D {r["pinn_D_err"]:+.2f}% k {r["pinn_k_err"]:+.2f}% | '
              f'ref D {r["ref_D_err"]:+.2f}% k {r["ref_k_err"]:+.2f}%  ({r["minutes"]:.1f} min)')

    # ---- summary ---------------------------------------------------------------
    R = [res['runs'][str(s)] for s in DRAW_SEEDS if str(s) in res['runs']]
    A = lambda k: np.array([r[k] for r in R])
    summ = {}
    for m in ('pinn', 'ref'):
        for q in ('D', 'k'):
            e = A(f'{m}_{q}_err')
            summ[f'{m}_{q}'] = {'mean': e.mean(), 'sd': e.std(ddof=1) if e.size > 1 else 0.0,
                                'rmse': np.sqrt(np.mean(e**2)), 'mean_abs': np.abs(e).mean()}
    summ['coverage_ref_D'] = A('ref_covers_D').mean()
    summ['coverage_ref_k'] = A('ref_covers_k').mean()
    summ['coverage_ref_joint'] = A('truth_in_ellipse_ref').mean()
    summ['pinn_in_ref95_D'] = A('pinn_in_ref95_D').mean()
    summ['pinn_in_ref95_k'] = A('pinn_in_ref95_k').mean()
    summ['pinn_in_ref95_joint'] = A('pinn_in_ellipse_ref').mean()
    summ['mean_laplace_sd_D_pct'] = 100 * A('ref_sd_logD').mean()
    summ['mean_laplace_sd_k_pct'] = 100 * A('ref_sd_logk').mean()
    summ['corr_pinn_ref_D'] = np.corrcoef(A('pinn_D_err'), A('ref_D_err'))[0, 1] if len(R) > 2 else None
    summ['corr_pinn_ref_k'] = np.corrcoef(A('pinn_k_err'), A('ref_k_err'))[0, 1] if len(R) > 2 else None
    res['summary'] = summ
    save_results(name, res)
    print(f'\n  {len(R)} draws        mean     SD   RMSE   (signed % error)')
    for m in ('pinn', 'ref'):
        for q in ('D', 'k'):
            s_ = summ[f'{m}_{q}']
            print(f'  {m:<4} {q}:  {s_["mean"]:+7.2f} {s_["sd"]:6.2f} {s_["rmse"]:6.2f}')
    print(f'  mean Laplace SD: D {summ["mean_laplace_sd_D_pct"]:.2f}%, k {summ["mean_laplace_sd_k_pct"]:.2f}%')
    print(f'  reference 95% coverage of truth: D {summ["coverage_ref_D"]:.0%}, '
          f'k {summ["coverage_ref_k"]:.0%}, joint {summ["coverage_ref_joint"]:.0%}')
    print(f'  PINN inside reference 95% region: D {summ["pinn_in_ref95_D"]:.0%}, '
          f'k {summ["pinn_in_ref95_k"]:.0%}, joint {summ["pinn_in_ref95_joint"]:.0%}')

    fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
    for j, q in enumerate(('D', 'k')):
        ax[j].plot(A(f'ref_{q}_err'), A(f'pinn_{q}_err'), 'o', mec='k', mew=0.5)
        lim = 1.1 * np.max(np.abs(np.concatenate([A(f'ref_{q}_err'), A(f'pinn_{q}_err')])))
        ax[j].plot([-lim, lim], [-lim, lim], 'k:', lw=0.8)
        ax[j].set_xlabel(f'reference estimator {q} error (%)'); ax[j].set_ylabel(f'PINN {q} error (%)')
        ax[j].set_title(f'{q}: per-draw errors ({len(R)} draws)'); ax[j].grid(alpha=0.3)
    ax[2].plot(A('ref_D_err'), A('ref_k_err'), 's', color='0.5', label='reference')
    ax[2].plot(A('pinn_D_err'), A('pinn_k_err'), 'o', color='C3', mec='k', mew=0.5, label='PINN')
    ax[2].axhline(0, color='k', lw=0.6); ax[2].axvline(0, color='k', lw=0.6)
    ax[2].set_xlabel('D error (%)'); ax[2].set_ylabel('k error (%)'); ax[2].legend(); ax[2].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT_DIR / f'{name}.png', dpi=150); plt.show()
    finish(name, [f'{name}.png'])

## E2. Truth-free selection of the data weight $W_D$

In the paper $W_D = 5$ was chosen by a rule that used the true parameters on a tuning draw.
A practitioner cannot do that. Two selection rules that use only the data are tested over
$W_D \in \{1, 3, 5, 10, 30, 100\}$:

* **Discrepancy principle** (Morozov): the data misfit should match the noise. Choose the
  smallest $W_D$ whose RMS data residual $\le \tau\sigma$ ($\tau = 1$, $\sigma = 2\%$ of $C_0$,
  assumed known as in the reference estimator). Uses the full-data runs.
* **K-fold cross-validation**: train on $K-1$ folds of the observations, predict the held-out
  fold; choose the $W_D$ with the lowest held-out MSE.

Discrepancy runs on the paper draw (4001) and two new draws (8001, 8002); cross-validation on
draw 4001. The full-data runs also give the $D$, $k$ errors at every $W_D$, so the table shows
whether each rule lands where the truth-based choice would.

In [ ]:
TF_WD    = [1.0, 5.0, 30.0] if QUICK else [1.0, 3.0, 5.0, 10.0, 30.0, 100.0]
TF_SEEDS = [4001] if QUICK else [4001, 8001, 8002]
TF_CV_SEED, TF_FOLDS, TF_TAU = 4001, (2 if QUICK else 4), 1.0

if not RUN_TRUTHFREE:
    print('RUN_TRUTHFREE = False, skipping.')
else:
    name = 'E2_truthfree_wd'
    res = load_results(name, {'W_D': TF_WD, 'obs_seeds': TF_SEEDS, 'cv_seed': TF_CV_SEED,
                              'folds': TF_FOLDS, 'tau': TF_TAU, 'net_seed': SEED})
    pts = sample_points()
    sig = NOISE_PCT / 100 * C0
    rep = lambda q: f'D {float(jnp.exp(q["logD"])):.4f} k {float(jnp.exp(q["logk"])):.5f}'

    def fit(obs, w_d, label):
        p0 = {'net': init_net(SEED), 'logD': jnp.log(D_INIT), 'logk': jnp.log(K_INIT)}
        return train(p0, build_loss(phys_linear, w_d=w_d), make_batch(pts, obs), label, rep)

    def predict_net(p, obs):
        f = net_fn(p)
        return np.asarray(jax.vmap(lambda x, y, t: f(jnp.stack([x, y, t])))(
            *(jnp.asarray(obs[k]) for k in ('x_d', 'y_d', 't_d'))))

    # ---- full-data runs: errors + discrepancy -------------------------------------
    for s in TF_SEEDS:
        obs = scattered_obs(FR_LIN, SOLVE_LIN, s)
        for w in TF_WD:
            key = f'full_s{s}_w{w:g}'
            if key in res['runs']:
                continue
            t0 = time.time()
            p = fit(obs, w, key)
            D_p, k_p = float(jnp.exp(p['logD'])), float(jnp.exp(p['logk']))
            resid = predict_net(p, obs) - obs['C_d']
            res['runs'][key] = {'seed': s, 'w_d': w, 'D_err': pct(D_p, D_TRUE), 'k_err': pct(k_p, K_TRUE),
                                'rms_over_sigma': float(np.sqrt(np.mean(resid**2)) / sig),
                                'minutes': (time.time() - t0) / 60}
            save_results(name, res)
            r = res['runs'][key]
            print(f'  {key}: D {r["D_err"]:+.2f}% k {r["k_err"]:+.2f}% RMS/sigma {r["rms_over_sigma"]:.3f}')

    # ---- cross-validation ---------------------------------------------------------
    obs = scattered_obs(FR_LIN, SOLVE_LIN, TF_CV_SEED)
    fold = np.random.default_rng(99).permutation(N_OBS) % TF_FOLDS
    for w in TF_WD:
        for f_ in range(TF_FOLDS):
            key = f'cv_w{w:g}_f{f_}'
            if key in res['runs']:
                continue
            tr = {k: v[fold != f_] for k, v in obs.items()}
            te = {k: v[fold == f_] for k, v in obs.items()}
            t0 = time.time()
            p = fit(tr, w, key)
            res['runs'][key] = {'w_d': w, 'fold': f_,
                                'heldout_mse': float(np.mean((predict_net(p, te) - te['C_d'])**2)),
                                'D_err': pct(float(jnp.exp(p['logD'])), D_TRUE),
                                'k_err': pct(float(jnp.exp(p['logk'])), K_TRUE),
                                'minutes': (time.time() - t0) / 60}
            save_results(name, res)
            print(f'  {key}: held-out MSE / sigma^2 = {res["runs"][key]["heldout_mse"] / sig**2:.3f}')

    # ---- selections ---------------------------------------------------------------
    R = res['runs']
    sel = {'discrepancy': {}, 'truth_best_maxabs': {}}
    print(f'\n  {"seed":<6}{"W_D":>6}{"D err":>9}{"k err":>9}{"RMS/sigma":>11}')
    for s in TF_SEEDS:
        rows = [R[f'full_s{s}_w{w:g}'] for w in TF_WD if f'full_s{s}_w{w:g}' in R]
        for r in rows:
            print(f'  {s:<6}{r["w_d"]:>6g}{r["D_err"]:>+8.2f}%{r["k_err"]:>+8.2f}%{r["rms_over_sigma"]:>11.3f}')
        ok = [r for r in rows if r['rms_over_sigma'] <= TF_TAU]
        pick = min(ok, key=lambda r: r['w_d']) if ok else max(rows, key=lambda r: r['w_d'])
        best = min(rows, key=lambda r: max(abs(r['D_err']), abs(r['k_err'])))
        sel['discrepancy'][str(s)] = pick['w_d']
        sel['truth_best_maxabs'][str(s)] = best['w_d']
        print(f'  -> seed {s}: discrepancy picks W_D = {pick["w_d"]:g} '
              f'(D {pick["D_err"]:+.2f}%, k {pick["k_err"]:+.2f}%); '
              f'truth-based best W_D = {best["w_d"]:g}')
    cv = {w: np.mean([R[f'cv_w{w:g}_f{f_}']['heldout_mse'] for f_ in range(TF_FOLDS)
                      if f'cv_w{w:g}_f{f_}' in R]) for w in TF_WD}
    sel['cv_mse_over_sigma2'] = {str(w): v / sig**2 for w, v in cv.items()}
    sel['cv'] = min(cv, key=cv.get)
    print('  CV held-out MSE / sigma^2: ' + ', '.join(f'W_D={w:g}: {v / sig**2:.3f}' for w, v in cv.items()))
    print(f'  -> CV picks W_D = {sel["cv"]:g}')
    res['selection'] = sel
    save_results(name, res)

    fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
    for s in TF_SEEDS:
        rows = [R[f'full_s{s}_w{w:g}'] for w in TF_WD if f'full_s{s}_w{w:g}' in R]
        ws = [r['w_d'] for r in rows]
        ax[0].semilogx(ws, [r['D_err'] for r in rows], 'o-', label=f'D, draw {s}')
        ax[0].semilogx(ws, [r['k_err'] for r in rows], 's--', label=f'k, draw {s}')
        ax[1].semilogx(ws, [r['rms_over_sigma'] for r in rows], 'o-', label=f'draw {s}')
    ax[0].axhline(0, color='k', lw=0.6); ax[0].set_xlabel('$W_D$'); ax[0].set_ylabel('signed error (%)')
    ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
    ax[1].axhline(TF_TAU, color='k', ls=':', label=r'$\tau\sigma$')
    ax[1].set_xlabel('$W_D$'); ax[1].set_ylabel(r'RMS data residual / $\sigma$'); ax[1].legend(fontsize=7)
    ax[1].grid(alpha=0.3); ax[1].set_title('discrepancy principle')
    ax[2].semilogx(list(cv), [v / sig**2 for v in cv.values()], 'o-')
    ax[2].set_xlabel('$W_D$'); ax[2].set_ylabel(r'held-out MSE / $\sigma^2$')
    ax[2].set_title(f'{TF_FOLDS}-fold cross-validation (draw {TF_CV_SEED})'); ax[2].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT_DIR / f'{name}.png', dpi=150); plt.show()
    finish(name, [f'{name}.png'])

## E3. Michaelis-Menten uptake: when are $V_{max}$ and $K_m$ identifiable?

Linear uptake $kC$ is the low-concentration limit of saturable transport
$V_{max}C/(K_m + C)$ with $k = V_{max}/K_m$. Cragg & Rice (2004): $V_{max} = 4.1$ µM/s
$= 0.0041$ µM/ms, $K_m = 0.21$ µM, so $V_{max}/K_m = 0.0195$ ms$^{-1}$ (the paper's $k$).
The data are generated with MM uptake in two settings:

* **E3a, the paper's setting**: 50 ms record, $C_0 = 1$ µM, 2% noise. In 50 ms uptake removes
  only ~0.2 µM, so the concentration never falls through $K_m$; a pre-check with the reference
  estimator gave corr$(V_{max}, K_m) = 1.000$: only the ratio $V_{max}/K_m$ is identifiable, which
  is what the linear model estimates.
* **E3b, a clearance-length record**: 500 ms, $C_0 = 10$ µM, noise SD 0.02 µM (2% of 1 µM, typical
  of an electrode). The concentration now crosses $K_m$; the pre-check gave Laplace SDs of 3%
  for $D$, 3% for $V_{max}$ and 7% for $K_m$, and the linear model is rejected by its residuals.
  For the PINN, time is scaled by 50/500 at the network input so it sees the paper's range,
  and interior points are denser at early times ($t = 500\,u^2$ ms).

In both settings: MM fit and linear fit by the reference estimator (Laplace SDs, correlation,
AIC, $\chi^2$ test of the residuals) and by the PINN (MM: 3 network seeds in E3b, 1 in E3a;
linear: 1 seed). The linear fit to MM data shows what the paper's model reports when the uptake is
really saturable.

In [ ]:
VMAX_TRUE, KM_TRUE = 0.0041, 0.21      # uM/ms, uM
MM_OBS_SEED = 9001
MM_CASES = {   # name: (record length ms, peak uM, noise SD uM, dt_store ms, PINN MM seeds)
    'E3a_T50_C1':   (50.0, 1.0, 0.02, DT_STORE, [SEED]),
    'E3b_T500_C10': (500.0, 10.0, 0.02, 0.5 if not QUICK else 1.0, [SEED] if QUICK else [1234, 1235, 1236]),
}

if not RUN_MM:
    print('RUN_MM = False, skipping.')
else:
    name = 'E3_michaelis_menten'
    res = load_results(name, {'D': D_TRUE, 'Vmax': VMAX_TRUE, 'Km': KM_TRUE, 'obs_seed': MM_OBS_SEED,
                              'cases': {k: v[:4] for k, v in MM_CASES.items()}})
    mm_sink = lambda Vm, Km: (lambda C: Vm * C / (Km + C))
    rep_mm = lambda q: (f'D {float(jnp.exp(q["logD"])):.4f} Vmax {float(jnp.exp(q["logVmax"])):.5f} '
                        f'Km {float(jnp.exp(q["logKm"])):.4f}')
    rep_lin = lambda q: f'D {float(jnp.exp(q["logD"])):.4f} k {float(jnp.exp(q["logk"])):.5f}'
    case_data = {}
    for cname, (T_end, amp, sig, dts, seeds) in MM_CASES.items():
        print('=' * 72); print(f'  {cname}: T = {T_end:g} ms, C0 = {amp:g} uM, noise SD {sig} uM'); print('=' * 72)
        solver = make_fd_solver(D_cap=0.64, T_end=T_end, dt_store=dts)
        Ci = gaussian_ic(solver.nx, amp)
        FR = solver(D_TRUE, mm_sink(VMAX_TRUE, KM_TRUE), Ci)
        t_obs = (0.5, 0.9 * T_end)
        obs = scattered_obs(FR, solver, MM_OBS_SEED, noise_pct=100 * sig / C0, t_obs=t_obs)
        case_data[cname] = obs
        x_, y_, t_ = (jnp.asarray(obs[k]) for k in ('x_d', 'y_d', 't_d'))
        t_scale = T / T_end

        # ---- reference estimator: MM and linear model ------------------------
        rk = f'{cname}_ref'
        if rk not in res:
            pr_mm = lambda th: interp_frames(solver(jnp.exp(th[0]), mm_sink(jnp.exp(th[1]), jnp.exp(th[2])), Ci),
                                             solver.dx, x_, y_, t_, dts)
            rf = ref_fit(pr_mm, np.log([D_INIT, 0.8 * VMAX_TRUE, 1.2 * KM_TRUE]), obs['C_d'], sig,
                         np.log([0.02, 1e-5, 1e-3]), np.log([0.62, 0.5, 50.0]))
            rl = lin_ref_fit(obs, solver, amp=amp)
            sd = np.sqrt(np.diag(rf['cov']))
            D_, Vm_, Km_ = np.exp(rf['theta'])
            res[rk] = {
                'mm': {'D_err': pct(D_, D_TRUE), 'Vmax_err': pct(Vm_, VMAX_TRUE), 'Km_err': pct(Km_, KM_TRUE),
                       'Vmax_over_Km_err': pct(Vm_ / Km_, VMAX_TRUE / KM_TRUE),
                       'sd_pct': (100 * sd).tolist(), 'corr': (rf['cov'] / np.outer(sd, sd)).tolist(),
                       'rms_over_sigma': rf['rms'] / sig, 'p_chi2': rf['p_chi2'], 'aic': rf['aic'],
                       'at_bound': rf['at_bound']},
                'linear': {'D_err': pct(np.exp(rl['theta'][0]), D_TRUE), 'k_eff': float(np.exp(rl['theta'][1])),
                           'k_eff_over_Vmax_Km': float(np.exp(rl['theta'][1]) / (VMAX_TRUE / KM_TRUE)),
                           'sd_pct': (100 * np.sqrt(np.diag(rl['cov']))).tolist(),
                           'rms_over_sigma': rl['rms'] / sig, 'p_chi2': rl['p_chi2'], 'aic': rl['aic']},
                'obs_C_percentiles_5_50_95': np.percentile(obs['C_clean'], [5, 50, 95]).tolist()}
            save_results(name, res)
        r = res[rk]
        print(f'  ref MM    : D {r["mm"]["D_err"]:+.2f}%  Vmax {r["mm"]["Vmax_err"]:+.2f}%  Km {r["mm"]["Km_err"]:+.2f}%  '
              f'Vmax/Km {r["mm"]["Vmax_over_Km_err"]:+.2f}%')
        print('              Laplace SD (%) D, Vmax, Km: ' + ', '.join(f'{v:.1f}' for v in r['mm']['sd_pct'])
              + f';  corr(Vmax, Km) = {r["mm"]["corr"][1][2]:+.3f}')
        print(f'  ref linear: D {r["linear"]["D_err"]:+.2f}%  k_eff {r["linear"]["k_eff"]:.5f} '
              f'({r["linear"]["k_eff_over_Vmax_Km"]:.2f} x Vmax/Km)  RMS/sigma {r["linear"]["rms_over_sigma"]:.3f}  '
              f'p(chi2) {r["linear"]["p_chi2"]:.3g}')
        print(f'  AIC(MM) - AIC(linear) = {r["mm"]["aic"] - r["linear"]["aic"]:+.1f}  (negative favours MM)')

        # ---- PINN ------------------------------------------------------------
        pts = sample_points(T_end=T_end, amp=amp, t_power=(2.0 if T_end > T else 1.0))
        batch = make_batch(pts, obs, t_scale)
        for s in seeds:
            key = f'{cname}_pinn_mm_s{s}'
            if key in res['runs']:
                continue
            t0 = time.time()
            p0 = {'net': init_net(s), 'logD': jnp.log(D_INIT),
                  'logVmax': jnp.log(0.8 * VMAX_TRUE), 'logKm': jnp.log(1.2 * KM_TRUE)}
            p = train(p0, build_loss(phys_mm), batch, key, rep_mm)
            D_, Vm_, Km_ = (float(jnp.exp(p[k])) for k in ('logD', 'logVmax', 'logKm'))
            res['runs'][key] = {'case': cname, 'seed': s, 'D_err': pct(D_, D_TRUE), 'Vmax_err': pct(Vm_, VMAX_TRUE),
                                'Km_err': pct(Km_, KM_TRUE), 'Vmax_over_Km_err': pct(Vm_ / Km_, VMAX_TRUE / KM_TRUE),
                                'minutes': (time.time() - t0) / 60}
            save_results(name, res)
            q = res['runs'][key]
            print(f'  {key}: D {q["D_err"]:+.2f}% Vmax {q["Vmax_err"]:+.2f}% Km {q["Km_err"]:+.2f}% '
                  f'Vmax/Km {q["Vmax_over_Km_err"]:+.2f}%')
        key = f'{cname}_pinn_linear'
        if key not in res['runs']:
            t0 = time.time()
            p0 = {'net': init_net(SEED), 'logD': jnp.log(D_INIT), 'logk': jnp.log(K_INIT)}
            loss = build_loss(phys_linear)
            p = train(p0, loss, batch, key, rep_lin)
            L_r, L_i, L_b, L_d = (float(v) for v in loss.components(p, batch))
            res['runs'][key] = {'case': cname, 'D_err': pct(float(jnp.exp(p['logD'])), D_TRUE),
                                'k_eff': float(jnp.exp(p['logk'])), 'L_r': L_r,
                                'rms_data_over_sigma': float(np.sqrt(L_d) / sig), 'minutes': (time.time() - t0) / 60}
            save_results(name, res)
        q = res['runs'][key]
        print(f'  {key}: D {q["D_err"]:+.2f}%, k_eff {q["k_eff"]:.5f}, RMS data / sigma '
              f'{q["rms_data_over_sigma"]:.3f}')

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    C_grid = np.linspace(0, 2, 300)
    ax[0].plot(C_grid, VMAX_TRUE * C_grid / (KM_TRUE + C_grid), 'k-', lw=2, label='true MM')
    ax[0].plot(C_grid, VMAX_TRUE / KM_TRUE * C_grid, 'k:', label='linear, k = Vmax/Km')
    for j, cname in enumerate(MM_CASES):
        for key, q in res['runs'].items():
            if q.get('case') == cname and 'Vmax_err' in q:
                Vm_, Km_ = VMAX_TRUE * (1 + q['Vmax_err'] / 100), KM_TRUE * (1 + q['Km_err'] / 100)
                ax[0].plot(C_grid, Vm_ * C_grid / (Km_ + C_grid), f'C{j}-', alpha=0.6, lw=1)
        ax[0].plot([], [], f'C{j}-', label=f'PINN MM fits, {cname}')
        ax[1].hist(case_data[cname]['C_clean'], bins=40, alpha=0.5, color=f'C{j}', label=cname, log=True)
    ax[0].set_ylim(0, 0.006); ax[0].set_xlabel(r'C ($\mu$M)'); ax[0].set_ylabel(r'uptake ($\mu$M/ms)')
    ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
    ax[1].axvline(KM_TRUE, color='k', ls='--', label='$K_m$'); ax[1].set_xscale('symlog', linthresh=0.01)
    ax[1].set_xlabel(r'observed C ($\mu$M)'); ax[1].legend(fontsize=7); ax[1].set_title('data span relative to $K_m$')
    names = list(MM_CASES); xi = np.arange(3)
    for j, cname in enumerate(names):
        ax[2].bar(xi + (j - 0.5) * 0.35, res[f'{cname}_ref']['mm']['sd_pct'], 0.35, color=f'C{j}', label=cname)
    ax[2].set_yscale('log'); ax[2].set_xticks(xi); ax[2].set_xticklabels(['D', '$V_{max}$', '$K_m$'])
    ax[2].set_ylabel('reference Laplace SD (%)'); ax[2].legend(fontsize=7); ax[2].grid(alpha=0.3, axis='y')
    fig.tight_layout(); fig.savefig(OUT_DIR / f'{name}.png', dpi=150); plt.show()
    finish(name, [f'{name}.png'])

## E4. Spatially varying diffusion coefficient $D(x, y)$

Truth: $D(x, y) = D_0\,[1 - A\exp(-|{\bf x} - {\bf x}_c|^2 / 2w^2)]$ with $D_0 = 0.32$,
$A = 0.5$, ${\bf x}_c = (0.8, 0.4)$ µm, $w = 0.6$ µm: a patch of hindered diffusion next to the
release site. $k = 0.020$. The PDE becomes $\partial_t C = \nabla\cdot(D\nabla C) - kC$.

* **PINN**: a second network $[2, 32, 32, 1]$ gives $D(x, y) = 0.30\,e^{d(x, y)}$ (last layer
  initialised to zero, so it starts from a uniform $D = 0.30$). Runs with and without a weak
  smoothness penalty $W_S\,\overline{|\nabla d|^2}$.
* **Reference estimators** through the FD solver: (i) the true functional form with 5 shape
  parameters plus $k$ (best case: the form is known); (ii) a generic $5\times 5$ bilinear grid
  of $\log D$ plus $k$ (26 parameters).

**Observation design.** A pre-check with the reference estimator showed that the paper's design
(400 points over 0.5-45 ms) carries almost no information on the field: even with the true
functional form the error was 15.9%, the same as a constant $D$ (16.2%), because by then the
profile is nearly flat. Spatial structure of $D$ is visible only while gradients are steep, but
$k$ needs the late decay. The design used here is 600 points over 0.2-10 ms plus 400 over
0.5-45 ms (pre-check: field error 7.8%, $k$ -2.6%). The reference fit on the paper's design is
also reported. Error is the relative $L_2$ error of $D$ over $r < 1.5$ µm, where the solution
carries information about $D$, and over the whole square.

In [ ]:
DF_A, DF_XC, DF_YC, DF_W = 0.5, 0.8, 0.4, 0.6
DF_N_EARLY, DF_T_EARLY = (200 if QUICK else 600), (0.2, 10.0)
DF_SMOOTH = [0.0, 1e-3]
DF_OBS_SEED = 9101


def d_true_fn(x, y, D0=D_TRUE, A=DF_A, xc=DF_XC, yc=DF_YC, w=DF_W):
    return D0 * (1 - A * jnp.exp(-((x - xc)**2 + (y - yc)**2) / (2 * w**2)))


if not RUN_DFIELD:
    print('RUN_DFIELD = False, skipping.')
else:
    name = 'E4_D_field'
    res = load_results(name, {'D0': D_TRUE, 'A': DF_A, 'xc': DF_XC, 'yc': DF_YC, 'w': DF_W, 'k': K_TRUE,
                              'design': f'{DF_N_EARLY} pts in {DF_T_EARLY} ms + {N_OBS} pts in {T_OBS} ms',
                              'obs_seed': DF_OBS_SEED})
    nx = SOLVE_LIN.nx
    xg = np.linspace(-L / 2, L / 2, nx)
    XG, YG = np.meshgrid(xg, xg, indexing='ij')
    XGj, YGj = jnp.asarray(XG), jnp.asarray(YG)
    D_true_grid = np.asarray(d_true_fn(XGj, YGj))
    Ci = gaussian_ic(nx)
    FR_DF = SOLVE_LIN(D_true_grid, lambda C: K_TRUE * C, Ci)
    o1 = scattered_obs(FR_DF, SOLVE_LIN, DF_OBS_SEED, n_obs=DF_N_EARLY, t_obs=DF_T_EARLY)
    o2 = scattered_obs(FR_DF, SOLVE_LIN, DF_OBS_SEED + 1)
    obs = {k: np.concatenate([o1[k], o2[k]]) for k in o1}
    obs_paper = scattered_obs(FR_DF, SOLVE_LIN, DF_OBS_SEED + 2)     # the paper's design, for comparison
    sig = NOISE_PCT / 100 * C0
    inner = (XG**2 + YG**2) < 1.5**2

    def field_err(Dg):
        e = lambda m: float(np.sqrt(np.sum((Dg - D_true_grid)[m]**2) / np.sum(D_true_grid[m]**2)) * 100)
        return {'rel_L2_inner_pct': e(inner), 'rel_L2_all_pct': e(np.ones_like(inner))}

    x_, y_, t_ = (jnp.asarray(obs[k]) for k in ('x_d', 'y_d', 't_d'))
    if 'baseline_constant' not in res:
        res['baseline_constant'] = field_err(np.full_like(D_true_grid, D_true_grid[inner].mean()))
    for kk, o in (('ref_param', obs), ('ref_param_paper_design', obs_paper)):
        if kk in res:
            continue
        xo_, yo_, to_ = (jnp.asarray(o[k]) for k in ('x_d', 'y_d', 't_d'))
        def pred_param(th, xo_=xo_, yo_=yo_, to_=to_):   # th = log D0, A, xc, yc, log w, log k
            Dg = d_true_fn(XGj, YGj, jnp.exp(th[0]), th[1], th[2], th[3], jnp.exp(th[4]))
            return interp_frames(SOLVE_LIN(Dg, lambda C: jnp.exp(th[5]) * C, Ci), SOLVE_LIN.dx, xo_, yo_, to_)
        th0 = np.array([np.log(D_INIT), 0.2, 0.0, 0.0, np.log(1.0), np.log(K_INIT)])
        lb = np.array([np.log(0.05), 0.0, -2.0, -2.0, np.log(0.2), np.log(1e-4)])
        ub = np.array([np.log(0.62), 0.95, 2.0, 2.0, np.log(3.0), np.log(0.5)])
        rf = ref_fit(pred_param, th0, o['C_d'], sig, lb, ub)
        th = rf['theta']
        Dg = np.asarray(d_true_fn(XGj, YGj, np.exp(th[0]), th[1], th[2], th[3], np.exp(th[4])))
        res[kk] = {**field_err(Dg), 'k_err': pct(np.exp(th[5]), K_TRUE), 'theta': th.tolist(),
                   'sd': np.sqrt(np.diag(rf['cov'])).tolist(), 'rms_over_sigma': rf['rms'] / sig,
                   'D_grid': Dg.tolist()}
        save_results(name, res)
    print(f'  constant-D baseline: rel L2 inner {res["baseline_constant"]["rel_L2_inner_pct"]:.1f}%')
    if 'ref_grid' not in res:
        G = 3 if QUICK else 5
        gnodes = np.linspace(-L / 2, L / 2, G)
        # bilinear interpolation weights from the G x G node grid to the FD grid
        Wx = np.maximum(0, 1 - np.abs(xg[:, None] - gnodes[None, :]) / (gnodes[1] - gnodes[0]))
        Wxj = jnp.asarray(Wx)
        def pred_grid(th):
            Dg = jnp.exp(Wxj @ th[:-1].reshape(G, G) @ Wxj.T)
            return interp_frames(SOLVE_LIN(Dg, lambda C: jnp.exp(th[-1]) * C, Ci), SOLVE_LIN.dx, x_, y_, t_)
        th0 = np.concatenate([np.full(G * G, np.log(D_INIT)), [np.log(K_INIT)]])
        lb = np.concatenate([np.full(G * G, np.log(0.03)), [np.log(1e-4)]])
        ub = np.concatenate([np.full(G * G, np.log(0.62)), [np.log(0.5)]])
        rf = ref_fit(pred_grid, th0, obs['C_d'], sig, lb, ub)
        Dg = np.exp(Wx @ rf['theta'][:-1].reshape(G, G) @ Wx.T)
        res['ref_grid'] = {**field_err(Dg), 'k_err': pct(np.exp(rf['theta'][-1]), K_TRUE), 'G': G,
                           'rms_over_sigma': rf['rms'] / sig, 'D_grid': Dg.tolist()}
        save_results(name, res)
    for kk in ('ref_param_paper_design', 'ref_param', 'ref_grid'):
        q = res[kk]
        print(f'  {kk:<22}: D rel L2 inner {q["rel_L2_inner_pct"]:.1f}%, all {q["rel_L2_all_pct"]:.1f}%, '
              f'k {q["k_err"]:+.2f}%, RMS/sigma {q["rms_over_sigma"]:.3f}')

    pts = sample_points()
    batch = make_batch(pts, obs)
    for ws in DF_SMOOTH:
        key = f'pinn_ws{ws:g}'
        if key in res['runs']:
            continue
        smooth = None if ws == 0 else (lambda p, b, ws=ws: ws * jnp.mean(jax.vmap(
            lambda x, y: jnp.sum(jax.grad(lambda xy: jnp.log(d_field(p)(xy)))(jnp.stack([x, y]))**2))(
            b['x_r'], b['y_r'])))
        p0 = {'net': init_net(SEED), 'dnet': nnx.split(MLP(D_LAYERS, rngs=nnx.Rngs(SEED + 1), zero_last=True))[1],
              'logk': jnp.log(K_INIT)}
        t0 = time.time()
        Dsite = lambda q: float(d_field(q)(jnp.array([DF_XC, DF_YC])))
        p = train(p0, build_loss(phys_dfield, extra=smooth), batch, key,
                  lambda q: f'k {float(jnp.exp(q["logk"])):.5f} D(xc) {Dsite(q):.4f}')
        Dg = np.asarray(jax.vmap(jax.vmap(lambda x, y: d_field(p)(jnp.stack([x, y]))))(XGj, YGj))
        res['runs'][key] = {**field_err(Dg), 'k_err': pct(float(jnp.exp(p['logk'])), K_TRUE), 'W_S': ws,
                            'D_grid': Dg.tolist(), 'minutes': (time.time() - t0) / 60}
        save_results(name, res)
        q = res['runs'][key]
        print(f'  {key}: D rel L2 inner {q["rel_L2_inner_pct"]:.1f}%, all {q["rel_L2_all_pct"]:.1f}%, '
              f'k {q["k_err"]:+.2f}%')

    panels = [('true', D_true_grid)] + [(k, np.array(v['D_grid'])) for k, v in res['runs'].items()] + \
             [('ref, true form', np.array(res['ref_param']['D_grid'])),
              (f'ref, {res["ref_grid"]["G"]}x{res["ref_grid"]["G"]} grid', np.array(res['ref_grid']['D_grid']))]
    fig, ax = plt.subplots(1, len(panels) + 1, figsize=(3.3 * (len(panels) + 1), 3.2))
    for a, (t_, Dg) in zip(ax, panels):
        im = a.pcolormesh(XG, YG, Dg, vmin=0.12, vmax=0.36, cmap='viridis', shading='auto')
        a.add_patch(plt.Circle((0, 0), 1.5, fill=False, color='w', ls='--', lw=0.8))
        a.set_title(t_, fontsize=9); a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([])
    fig.colorbar(im, ax=ax[len(panels) - 1], fraction=0.046)
    j = np.argmin(np.abs(xg - DF_YC))
    for t_, Dg in panels:
        ax[-1].plot(xg, Dg[:, j], 'k-' if t_ == 'true' else '-', lw=1.5 if t_ == 'true' else 1, label=t_)
    ax[-1].set_xlabel(f'x (um) at y = {xg[j]:.2f}'); ax[-1].set_ylabel('D'); ax[-1].legend(fontsize=6)
    ax[-1].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT_DIR / f'{name}.png', dpi=150); plt.show()
    finish(name, [f'{name}.png'])

## E5. Extracellular space with cellular obstacles (tortuosity)

Eight impermeable discs (cells) occupy ~22% of the square, leaving an extracellular
volume fraction $\alpha \approx 0.78$; release is at the origin, clear of the discs. Dopamine
diffuses in the gaps with the free coefficient $D_{free} = 0.763$ µm²/ms (Cragg & Rice 2004) and
is taken up with $k = 0.020$ ms$^{-1}$; zero flux on the walls and on every disc. Data come
from a finer FD grid (`NX_OBS_FD = 161`, staircase discs, observations at extracellular grid
nodes) than the reference estimator would need, so the PINN, which sees the exact circles, faces a
realistic small geometry error.

* **PINN with the geometry**: collocation only in the extracellular space, zero-flux points on
  the disc boundaries; recovers $D_{free}$ and $k$ (2 network seeds).
* **Reference estimator with the geometry**: same FD grid (inverse crime, benchmark).
* **Homogeneous fits** (PINN and reference, no obstacles): the usual model of the paper applied
  to tortuous data. The fitted $D$ is an apparent $D^*$, and $\lambda_{app} = (D_{free}/D^*)^{1/2}$
  is the tortuosity it implies. This is the $D^* = D/\lambda^2$ relation behind the paper's
  $D = 0.32$.

In [ ]:
OBSTACLES = [(1.5, 0.2, 0.55), (-1.4, 0.9, 0.5), (0.3, 1.6, 0.5), (-0.6, -1.6, 0.55),
             (1.2, -1.5, 0.45), (-1.7, -0.6, 0.45), (1.8, 1.7, 0.4), (-1.8, 1.9, 0.35)]
D_FREE = 0.763
NX_OBS_FD = 61 if QUICK else 161
OB_SEEDS = [SEED] if QUICK else [1234, 1235]
OB_OBS_SEED = 9201
N_OBSTACLE_BC = 400 if QUICK else 2000

if not RUN_OBSTACLES:
    print('RUN_OBSTACLES = False, skipping.')
else:
    name = 'E5_obstacles'
    # geometry checks: discs disjoint, inside the walls, clear of the release site
    for i, (a, b, R) in enumerate(OBSTACLES):
        assert abs(a) + R < L / 2 - 0.05 and abs(b) + R < L / 2 - 0.05, f'disc {i} touches a wall'
        assert np.hypot(a, b) - R > 0.9, f'disc {i} too close to the release site'
        for a2, b2, R2 in OBSTACLES[i + 1:]:
            assert np.hypot(a - a2, b - b2) > R + R2 + 0.1, 'discs overlap'
    alpha = 1 - sum(np.pi * R**2 for _, _, R in OBSTACLES) / L**2
    res = load_results(name, {'obstacles': OBSTACLES, 'alpha': alpha, 'D_free': D_FREE, 'k': K_TRUE,
                              'nx': NX_OBS_FD, 'obs_seed': OB_OBS_SEED})
    print(f'  extracellular volume fraction alpha = {alpha:.3f}')

    xo = np.linspace(-L / 2, L / 2, NX_OBS_FD)
    XO, YO = np.meshgrid(xo, xo, indexing='ij')
    outside = lambda x, y: np.all([(x - a)**2 + (y - b)**2 > R**2 for a, b, R in OBSTACLES], axis=0)
    mask = outside(XO, YO)
    SOLVE_OB = make_fd_solver(NX_OBS_FD, D_cap=1.25, mask=mask)
    Ci_ob = gaussian_ic(NX_OBS_FD)
    t0 = time.time()
    FR_OB = jax.block_until_ready(SOLVE_OB(D_FREE, lambda C: K_TRUE * C, Ci_ob))
    print(f'  obstacle FD solve: {time.time() - t0:.1f} s')

    # observations at extracellular grid nodes (exact in space, linear in time)
    rng = np.random.default_rng(OB_OBS_SEED)
    fluid = np.argwhere(mask)
    sel = fluid[rng.choice(len(fluid), N_OBS, replace=False)]
    ox, oy = xo[sel[:, 0]], xo[sel[:, 1]]
    ot = rng.uniform(*T_OBS, N_OBS)
    clean = np.asarray(interp_frames(FR_OB, SOLVE_OB.dx, jnp.asarray(ox), jnp.asarray(oy), jnp.asarray(ot)))
    obs = {'x_d': ox, 'y_d': oy, 't_d': ot, 'C_clean': clean,
           'C_d': clean + rng.normal(0, NOISE_PCT / 100 * C0, N_OBS)}
    sig = NOISE_PCT / 100 * C0

    if 'ref' not in res:
        rf_geo = ref_fit(lin_predict(obs, SOLVE_OB), np.log([0.7, K_INIT]), obs['C_d'], sig,
                         np.log([0.02, 1e-4]), np.log([1.2, 0.5]))
        SOLVE_HOM = make_fd_solver(NX_FD, D_cap=1.25)
        rf_hom = ref_fit(lin_predict(obs, SOLVE_HOM), np.log([0.7, K_INIT]), obs['C_d'], sig,
                         np.log([0.02, 1e-4]), np.log([1.2, 0.5]))
        Dh = float(np.exp(rf_hom['theta'][0]))
        res['ref'] = {'geometry': {'D_err': pct(np.exp(rf_geo['theta'][0]), D_FREE),
                                   'k_err': pct(np.exp(rf_geo['theta'][1]), K_TRUE),
                                   'sd_pct': (100 * np.sqrt(np.diag(rf_geo['cov']))).tolist(),
                                   'rms_over_sigma': rf_geo['rms'] / sig, 'aic': rf_geo['aic']},
                      'homogeneous': {'D_app': Dh, 'k_err': pct(np.exp(rf_hom['theta'][1]), K_TRUE),
                                      'lambda_app': float(np.sqrt(D_FREE / Dh)),
                                      'rms_over_sigma': rf_hom['rms'] / sig, 'p_chi2': rf_hom['p_chi2'],
                                      'aic': rf_hom['aic']}}
        save_results(name, res)
    r = res['ref']
    print(f'  ref, with geometry : D_free {r["geometry"]["D_err"]:+.2f}%  k {r["geometry"]["k_err"]:+.2f}%  '
          f'(Laplace SD {r["geometry"]["sd_pct"][0]:.1f}%, {r["geometry"]["sd_pct"][1]:.1f}%)')
    print(f'  ref, homogeneous   : D* = {r["homogeneous"]["D_app"]:.4f}  -> lambda_app = '
          f'{r["homogeneous"]["lambda_app"]:.3f}; k {r["homogeneous"]["k_err"]:+.2f}%; '
          f'RMS/sigma {r["homogeneous"]["rms_over_sigma"]:.3f}, p(chi2) {r["homogeneous"]["p_chi2"]:.3g}')

    rep = lambda q: f'D {float(jnp.exp(q["logD"])):.4f} k {float(jnp.exp(q["logk"])):.5f}'
    pts_geo = sample_points(obstacles=OBSTACLES, n_obstacle_bc=N_OBSTACLE_BC)
    pts_hom = sample_points()
    for s in OB_SEEDS:
        key = f'pinn_geometry_s{s}'
        if key in res['runs']:
            continue
        t0 = time.time()
        p0 = {'net': init_net(s), 'logD': jnp.log(0.7), 'logk': jnp.log(K_INIT)}
        p = train(p0, build_loss(phys_linear), make_batch(pts_geo, obs), key, rep)
        res['runs'][key] = {'D_err': pct(float(jnp.exp(p['logD'])), D_FREE),
                            'k_err': pct(float(jnp.exp(p['logk'])), K_TRUE), 'minutes': (time.time() - t0) / 60}
        save_results(name, res)
        print(f'  {key}: D_free {res["runs"][key]["D_err"]:+.2f}%, k {res["runs"][key]["k_err"]:+.2f}%')
    key = 'pinn_homogeneous'
    if key not in res['runs']:
        t0 = time.time()
        p0 = {'net': init_net(SEED), 'logD': jnp.log(0.7), 'logk': jnp.log(K_INIT)}
        loss = build_loss(phys_linear)
        b = make_batch(pts_hom, obs)
        p = train(p0, loss, b, key, rep)
        Dh = float(jnp.exp(p['logD']))
        L_r, L_i, L_b, L_d = (float(v) for v in loss.components(p, b))
        res['runs'][key] = {'D_app': Dh, 'lambda_app': float(np.sqrt(D_FREE / Dh)),
                            'k_err': pct(float(jnp.exp(p['logk'])), K_TRUE),
                            'rms_data_over_sigma': float(np.sqrt(L_d) / sig), 'minutes': (time.time() - t0) / 60}
        save_results(name, res)
    q = res['runs'][key]
    print(f'  PINN homogeneous: D* = {q["D_app"]:.4f} -> lambda_app = {q["lambda_app"]:.3f}, '
          f'k {q["k_err"]:+.2f}%, RMS data/sigma {q["rms_data_over_sigma"]:.3f}')

    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    for a, tt in zip(ax, (2.0, 10.0, 40.0)):
        Cm = np.where(mask, np.asarray(FR_OB[int(round(tt / DT_STORE))]), np.nan)
        im = a.pcolormesh(XO, YO, Cm, shading='auto', cmap='viridis')
        for (xc, yc, R) in OBSTACLES:
            a.add_patch(plt.Circle((xc, yc), R, fill=False, color='w', lw=0.8))
        a.set_title(f't = {tt:g} ms'); a.set_aspect('equal'); fig.colorbar(im, ax=a, fraction=0.046)
    ax[0].plot(obs['x_d'], obs['y_d'], 'r.', ms=1.5)
    fig.tight_layout(); fig.savefig(OUT_DIR / f'{name}.png', dpi=150); plt.show()
    finish(name, [f'{name}.png'])

## Download everything

In [ ]:
import shutil
zip_path = shutil.make_archive(str(OUT_DIR.parent / 'dopamine_PINN_extensions_results'), 'zip', OUT_DIR)
print(f'Zipped results: {zip_path}')
try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except ImportError:
    print('(Not on Colab - results stay in', OUT_DIR, ')')